# Analytic verification of the asymptotic cascade analysis

Symbolic (machine-checked) verification of the identities and scaling laws behind the
threshold-/degree-/marginal-dominated regimes of $\kappa(z)$.

Each cell prints `True` when the symbolic identity holds.

In [1]:
import sympy as sp
sp.init_printing()
n, p = sp.symbols('n p', positive=True)
k = sp.symbols('k', positive=True, integer=True)
a, b = sp.symbols('a b', positive=True)
print('sympy', sp.__version__)

sympy 1.14.0


## 1. Beta-binomial pmf identity

$P(K{=}k)=\binom{n}{k}\frac{B(k+a,\,n-k+b)}{B(a,b)}$ is the Beta-mixture of Binomial.
The only nontrivial step is the Beta kernel $\int_0^1 p^{A-1}(1-p)^{B-1}dp=B(A,B)$.

In [2]:
np_, kp = sp.symbols('np kp', positive=True)
kernel = sp.integrate(p**(kp+a-1)*(1-p)**(np_-kp+b-1), (p,0,1), conds='none')
print('mixture kernel == B(k+a, n-k+b):',
      sp.simplify(kernel - sp.beta(kp+a, np_-kp+b)) == 0)

mixture kernel == B(k+a, n-k+b): True


## 2. Mixture moments

$\mathbb E[p^{q}]=\int_0^1 \frac{p^{q+a-1}(1-p)^{b-1}}{B(a,b)}dp=\frac{B(a+q,b)}{B(a,b)}$.

In [3]:
q = sp.symbols('q', positive=True)
mom = sp.integrate(p**(q+a-1)*(1-p)**(b-1), (p,0,1), conds='none')/sp.beta(a,b)
print('E[p^q] == B(a+q,b)/B(a,b):',
      sp.simplify(mom - sp.beta(a+q,b)/sp.beta(a,b)) == 0)

E[p^q] == B(a+q,b)/B(a,b): True


## 3. Stirling / Gamma-ratio limits

$\frac{n!}{(n-k)!}\to n^{k}$ and $\frac{\Gamma(n+c)}{\Gamma(n+d)}\to n^{c-d}$ (fixed $k$, $n\to\infty$).
Concrete $a=\tfrac34,\,b=3$; symbolic positive integer $k$.

In [4]:
a0, b0 = sp.Rational(3,4), sp.Integer(3)
L1 = sp.limit(sp.factorial(n)/sp.factorial(n-k)/n**k, n, sp.oo)
L2 = sp.limit(sp.gamma(n-k+b0)/(sp.gamma(n+a0+b0)*n**(-(k+a0))), n, sp.oo)
L3 = sp.limit(sp.gamma(n+1)/(sp.gamma(n+sp.Rational(15,4))*n**(1-sp.Rational(15,4))), n, sp.oo)
print('n!/(n-k)! ~ n^k                       :', L1)
print('Gamma(n-k+b)/Gamma(n+a+b) ~ n^-(k+a)  :', L2)
print('Gamma(n+c)/Gamma(n+d) ~ n^(c-d)       :', L3)

n!/(n-k)! ~ n^k                       : 1
Gamma(n-k+b)/Gamma(n+a+b) ~ n^-(k+a)  : 1
Gamma(n+c)/Gamma(n+d) ~ n^(c-d)       : 1


## 4. Fixed-$k$ large-$n$ pmf

Combining 1 & 3:
$P(k;n)\sim n^{-a}\dfrac{\Gamma(k+a)\,\Gamma(a+b)}{\Gamma(a)\,\Gamma(b)\,\Gamma(k+1)}$.

In [5]:
P = (sp.gamma(n+1)/(sp.gamma(k+1)*sp.gamma(n-k+1))
     * sp.gamma(k+a0)*sp.gamma(n-k+b0)/sp.gamma(n+a0+b0)/sp.beta(a0,b0))
lhs = sp.limit(n**a0 * P, n, sp.oo)
rhs = sp.gamma(k+a0)*sp.gamma(a0+b0)/(sp.gamma(k+1)*sp.gamma(a0)*sp.gamma(b0))
print('fixed-k large-n asymptotics:', sp.simplify(lhs-rhs) == 0)

fixed-k large-n asymptotics: True


## 5. Threshold-dominated regime ($\alpha_{th}<a+2$)

For $k\gg1$, $F(1/k)\sim C_{th}k^{-\alpha_{th}}$, so
$\sum_k k(k{-}1)P(k)F(1/k)\sim C_{th}\,\mathbb E[K^{2-\alpha_{th}}]\sim C_{th}n^{2-\alpha_{th}}\frac{B(a+2-\alpha_{th},b)}{B(a,b)}$,
using $\mathbb E[K^{s}]\sim n^{s}\mathbb E[p^{s}]$ for $s>0$ with the moment finite iff $\alpha_{th}<a+2$. With $z=nm_d$ this gives $\kappa(z)\sim C_1 z^{1-\alpha_{th}}$, $C_1=C_{th}\,m_d^{\alpha_{th}-2}\frac{B(a+2-\alpha_{th},b)}{B(a,b)}$.

In [6]:
th, z = sp.symbols('th z', positive=True)
md = a/(a+b)
from_sum = (1/(n*md))*th*n**(2-th)*sp.beta(a+2-th,b)/sp.beta(a,b)
in_z     = th*md**(th-2)*z**(1-th)*sp.beta(a+2-th,b)/sp.beta(a,b)
print('n=z/m_d bookkeeping -> C1 z^(1-alpha_th):',
      sp.simplify(from_sum.subs(n, z/md) - in_z) == 0)
print('finite moment iff alpha_th < a+2  (kernel p^(a+1-alpha_th) integrable):',
      sp.simplify(sp.beta(a+2-th,b)) != sp.oo)

n=z/m_d bookkeeping -> C1 z^(1-alpha_th): True
finite moment iff alpha_th < a+2  (kernel p^(a+1-alpha_th) integrable): True


## 6. Degree-dominated / marginal regimes

Summand $\sim k^{1+a-\alpha_{th}}$. Converges iff $\alpha_{th}>a+2$ (exponent $<-1$); at $\alpha_{th}=a+2$ it is $\sum 1/k\sim\ln n$, giving the $\ln z$ correction $\kappa\sim C_c\,z^{-(a+1)}\ln n$.

In [7]:
kk = sp.Symbol('k', positive=True, integer=True)
print('degree-dominated  sum k^-3/2  (a=3/4, alpha_th=3.25):',
      sp.summation(kk**sp.Rational(-3,2), (kk,2,sp.oo)))
print('marginal  sum 1/k  diverges -> log :', sp.summation(1/kk, (kk,2,sp.oo)))
print('H_n - ln n -> EulerGamma          :', sp.limit(sp.harmonic(n)-sp.log(n), n, sp.oo))

degree-dominated  sum k^-3/2  (a=3/4, alpha_th=3.25): zeta(3/2, 2)
marginal  sum 1/k  diverges -> log : oo
H_n - ln n -> EulerGamma          : EulerGamma


## 7. Independent numeric cross-check (asserts)

In [8]:
import numpy as np
from scipy.special import beta as B, gammaln, loggamma
a0n, b0n = 0.75, 3.0
nnum = 4000
kn = np.arange(0, nnum+1)
logp = (gammaln(nnum+1)-gammaln(kn+1)-gammaln(nnum-kn+1)
        +loggamma(kn+a0n)+loggamma(nnum-kn+b0n)+loggamma(a0n+b0n)
        -(loggamma(nnum+a0n+b0n)+loggamma(a0n)+loggamma(b0n)))
pk = np.exp(logp)
m = a0n/(a0n+b0n)
assert abs(pk.sum()-1) < 1e-9                                  # normalisation
assert abs((kn*pk).sum() - nnum*m) < 1e-6                      # E[K] = n m_d
assert abs((pk*kn*(kn-1)).sum() - nnum*(nnum-1)*B(a0n+2,b0n)/B(a0n,b0n)) < 1e-4  # E[K(K-1)]
print('all numeric asserts passed')

all numeric asserts passed
